In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("/workspaces/llm-zoomcamp-2026/.env")

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
#Q1 ANSWER
len(documents)

72

In [5]:
#Q2 ANSWER

import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query)
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [6]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-23 13:17:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.3’

rag_helper.py.3     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-23 13:17:33 (39.6 MB/s) - ‘rag_helper.py.3’ saved [2134/2134]



In [7]:
#Q3 ANSWER

In [12]:
from rag_helper import RAGBase

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results
        )

    def build_context(self, search_results):
        context = []

        for doc in search_results:
            context.append(f"filename: {doc['filename']}")
            context.append(doc["content"])
            context.append("")

        return "\n".join(context).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)

        response = self.llm(prompt)

        answer = response.output_text
        usage = response.usage

        return answer, usage

In [14]:
rag = LessonRAG(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [15]:
import tiktoken

query = "How does the agentic loop keep calling the model until it stops?"

search_results = rag.search(query)
prompt = rag.build_prompt(query, search_results)

enc = tiktoken.encoding_for_model("gpt-4o-mini")
input_tokens_estimate = len(enc.encode(prompt))

print(input_tokens_estimate)

7068


In [17]:
#Q4 ANSWER


from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [19]:
#Q5 ANSWER

from gitsource import chunk_documents
import minsearch
import tiktoken

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

chunk_rag = LessonRAG(
    index=chunk_index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)


In [20]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = chunk_rag.search(query)
prompt = chunk_rag.build_prompt(query, search_results)

enc = tiktoken.encoding_for_model("gpt-4o-mini")
chunk_tokens = len(enc.encode(prompt))

print(chunk_tokens)

2251


In [21]:
q3_tokens = 7000
print(q3_tokens / chunk_tokens)

3.109729009329187


In [23]:
#Q6 ANSWER

search_call_count = 0

def search(query: str) -> list[dict]:
    """
    Search the course lesson chunks for relevant information.
    """
    global search_call_count
    search_call_count += 1

    results = chunk_index.search(query, num_results=5)
    return results


print(search_call_count)

0
